# Phase 0 — EyeCLIP single-image test
Goal: load checkpoint → run one image → print shape / dtype / range.
Stop here. No prototype, no full pipeline.

## 1. Install + clone

In [1]:
%pip install -q setuptools

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q ftfy regex tqdm
%pip install -q git+https://github.com/openai/CLIP.git
!git clone --depth 1 https://github.com/Michi-3000/EyeCLIP.git
import sys
sys.path.insert(0, "EyeCLIP")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


fatal: destination path 'EyeCLIP' already exists and is not an empty directory.


## 2. Load model
Assumes `eyeclip_visual.pt` is already in the working directory.

In [3]:
import pathlib, re

clip_path = pathlib.Path(r"E:\rop\codes\EyeCLIP\eyeclip\clip.py")
text = clip_path.read_text()
text = text.replace(
    "from pkg_resources import packaging",
    "import packaging"
)
clip_path.write_text(text)
print("patched:", clip_path)

patched: E:\rop\codes\EyeCLIP\eyeclip\clip.py


In [4]:
%pip install -q packaging

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys
sys.path.insert(0, r"E:\rop\codes\EyeCLIP")

import eyeclip
print("ok", eyeclip)

import torch
import eyeclip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model, preprocess = eyeclip.load("ViT-B/32", device=device, jit=False)
ckpt = torch.load("eyeclip.pt", map_location=device)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"], strict=False)
else:
    model.load_state_dict(ckpt, strict=False)

model.eval()
print("model loaded")

ok <module 'eyeclip' from 'E:\\rop\\codes\\EyeCLIP\\eyeclip\\__init__.py'>
device: cpu
model loaded


## 3. One-image test
Change `IMG_PATH` to any real ROP-VL image.

In [8]:
IMG_PATH = r"E:\rop\dataset\ROP-VL\Image\Normal\1_2020-04-03_L_3.jpg"  # <-- change if needed

img = Image.open(IMG_PATH).convert("RGB")
x = preprocess(img).unsqueeze(0).to(device)  # (1, 3, 224, 224)

with torch.no_grad():
    emb = model.encode_image(x)

print("shape :", emb.shape)
print("dtype :", emb.dtype)
print("min   :", emb.min().item())
print("max   :", emb.max().item())
print("mean  :", emb.mean().item())
print("std   :", emb.std().item())


shape : torch.Size([1, 512])
dtype : torch.float32
min   : -4.5407891273498535
max   : 1.8633869886398315
mean  : 0.016598626971244812
std   : 0.42182686924934387


## Done
Record the printed shape (expected ~ (1, 512)).  
Do **not** continue to prototypes or full embedding extraction yet.